# 02_Data Proecessing
- `data/diabetes.csv`를 이용해 EDA를 진행
- EDA 를 기반으로 파악된 null, 0값을 대체
- 하기 결측 처리 전략에 따라 진행

> ## 데이터 특성에 따른 선택
> | 데이터 특성      | 추천 방법                       | 이유                    |
> | ----------- | --------------------------- | --------------------- |
> | **소규모 데이터** | Mean/Median, KNN(k=3-5)     | 계산 효율, 과적합 위험         |
> | **중규모 데이터** | MissForest, MICE, KNN       | 정확도와 계산 시간의 균형        |
> | **대규모 데이터** | Deep Learning, GAN          | 충분한 학습 데이터, 복잡한 패턴 학습 |
> | **시계열 데이터** | LOCF, 보간, Transformer       | 시간적 의존성 고려            |
> | **혼합 데이터**  | MissForest, MICE, GAN       | 연속형 + 범주형 동시 처리       |
> | **고차원 데이터** | NMF, Matrix Completion, VAE | 차원 축소, 구조 학습          |
> | **비음수 데이터** | NMF                         | 비음수 제약 자연스러움          |
>
> ---
>
> 결측 비율에 따른 선택
>
> 낮은 결측률 (< 5%)
> - **1순위**: Mean/Median Imputation (충분히 좋은 성능)
> - **2순위**: KNN (약간 더 정확)
> - **3순위**: MICE
>
> 중간 결측률 (5-20%)
> - **1순위**: MissForest, MICE, KNN
> - **2순위**: Deep Learning (데이터 충분하면)
> - 피해야 할 것: Simple deletion
> 
> 높은 결측률 (> 20%, 특히 > 50%)
> - **1순위**: Deep Learning (GAN, VAE, DDPM)
> - **2순위**: Advanced Statistical (EM, BPCA)
> - **3순위**: Matrix Factorization


## 1. 환경 설정과 데이터 로드
- 필요한 라이브러리를 불러오고 `df1`이라는 이름으로 원본 데이터를 메모리에 적재
- 이후 단계에서 모든 분석은 `df1`을 기준으로, 데이터의 삭제, 추가가 진행될 경우 `df2`, `df3` 이런식으로 카운트가 올라가게 변수 진행

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from pyampute.exploration.mcar_statistical_tests import MCARTest
import numpy as np
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression


# from statsmodels.stats.missing import mcar_test

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_columns', None)
pd.set_option('future.no_silent_downcasting', True)


In [2]:
data_path = 'data/diabetes.csv'
df1 = pd.read_csv(data_path)

In [3]:
print(f'행/열 크기 : {df1.shape}')
print(f'칼럼명 : {df1.columns}')

행/열 크기 : (768, 9)
칼럼명 : Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')


## 2. Null Imputation을 위한 패턴파악 (참고)
- `NULL` 값의 경우 실제로 측정 했으나 측정이 한것으로 간주 
- 이전에 진행한 [EDA](./01_EDA.ipynb#4.-결측·이상-신호-점검) 결과에 따르면 전체 feature들 중에서 zero_count가 있는 변수는 `BMI`, `BloodPressure`, `Glucose`, `Insulin`, `SkinThickness` 가 존재함

### 2-1. Null 패턴 파악 (MCAR/MAR/MNAR)
- 최적의 Imputation 방법을 찾기 위해 어떤 패턴으로 결측치를 가지는지 파악
- EDA에서 0으로 기록된 값을 결측치로 간주한 뒤 결측 패턴 파악
    - 이전에 진행한 [EDA](./01_EDA.ipynb#4.-결측·이상-신호-점검)에서 의학적으로 나올 수 없는 변수들에 대해서만 0값을 NA로 간주
- 각 결측 패턴의 판정법과 해석 팁을 함께 적어 결과 해석에 바로 활용할 수 있도록 합니다.

In [4]:
# NA 변환 진행
# 변환 진행할 feature
target_cols = ['BMI', 'BloodPressure', 'Glucose', 'Insulin', 'SkinThickness']

# 분석/대체용 데이터 복제 및 0값을 결측으로 변환
df2 = df1.copy()
df2[target_cols] = df2[target_cols].replace(0, pd.NA)
df2_clean = df2.replace({pd.NA: np.nan}).astype(float) # Numpy 연산을 위한 NaN 변환

In [5]:
# 변환 여부 확인
print('결측치로 변환한 후 데이터 크기:', df2.shape)

display(df2[target_cols].head())

missing = df2.isna().sum().to_frame('missing_count')
zero_counts = (df2[target_cols] == 0).sum().to_frame('zero_count')
zero_counts['zero_ratio'] = (zero_counts['zero_count'] / len(df2)).round(3)
outlier_signals = missing.join(zero_counts, how='outer').fillna(0)

display(outlier_signals)

결측치로 변환한 후 데이터 크기: (768, 9)


,BMI,BloodPressure,Glucose,Insulin,SkinThickness
0,33.6,72,148,<NA>,35
1,26.6,66,85,<NA>,29
2,23.3,64,183,<NA>,<NA>
3,28.1,66,89,94,23
4,43.1,40,137,168,35


,missing_count,zero_count,zero_ratio
Age,0,0.0,0.0
BMI,11,0.0,0.0
BloodPressure,35,0.0,0.0
DiabetesPedigreeFunction,0,0.0,0.0
Glucose,5,0.0,0.0
Insulin,374,0.0,0.0
Outcome,0,0.0,0.0
Pregnancies,0,0.0,0.0
SkinThickness,227,0.0,0.0


- 확인 결과 빠짐없이 NA로 대체 되었음 확인 

In [6]:
# 결측 개수 및 비율 요약
missing_summary = pd.DataFrame({
    'missing_count': df2[target_cols].isna().sum(),
    'missing_percentage': df2[target_cols].isna().mean().round(3)
}).sort_values('missing_percentage', ascending=False)

display(missing_summary)

,missing_count,missing_percentage
Insulin,374,0.487
SkinThickness,227,0.296
BloodPressure,35,0.046
BMI,11,0.014
Glucose,5,0.007


### MCAR(Missing Completely At Random) 완전 무작위 결측 테스트 수행
- p-value가 0.05보다 크면 "결측이 완전무작위"라는 귀무가설을 유지해 MCAR 가능성이 높음 
- p-value가 0.05보다 작으면 MCAR가 아니므로 MAR 또는 MNAR 가능성을 고려

In [7]:
mt = MCARTest(method="little")
p_value = mt.little_mcar_test(df2_clean)

# 해석
alpha = 0.05
if p_value > alpha:
    print("P-value가 유의 수준(0.05)보다 크므로, 데이터가 MCAR이라는 귀무 가설을 기각할 수 없음 (MCAR일 가능성 높음)")
else:
    print("P-value가 유의 수준(0.05)보다 작으므로, 데이터가 MCAR이라는 귀무 가설을 기각. (MCAR 아닐 가능성 높음 - MAR 또는 MNAR)")

P-value가 유의 수준(0.05)보다 작으므로, 데이터가 MCAR이라는 귀무 가설을 기각. (MCAR 아닐 가능성 높음 - MAR 또는 MNAR)


- MCAR 가능성이 적은것으로 판별 후 추가적으로 결측 판별 진행

### MAR(Missing At Random)무작위 결측 판별
- 결측 여부(1=결측)를 다른 관측 변수와의 상관으로 진단
- 결측 여부가 다른 관측 변수(예: `Pregnancies`, `Age`, `Outcome` 등)와 상관을 보이면 MAR 가능성이 높음. 
- 상관 절대값이 0.1 이상인 항목 확인

In [8]:
mar_corr = {}
for col in target_cols:
    indicator = df2[col].isna().astype(int)
    corr_vals = {}
    for other in df2.columns:
        if other == col:
            continue
        corr_vals[other] = indicator.corr(df2[other].fillna(df2[other].median()))
    mar_corr[col] = pd.Series(corr_vals).sort_values(key=lambda s: s.abs(), ascending=False).head(3)

mar_corr_df = pd.concat(mar_corr, axis=1)
print('결측 여부와 가장 강하게 연관된 상위 3개 변수 (절대상관도 기준):')
mar_corr_df

결측 여부와 가장 강하게 연관된 상위 3개 변수 (절대상관도 기준):


,BMI,BloodPressure,Glucose,Insulin,SkinThickness
Glucose,-0.068889,NaN,NaN,NaN,NaN
Outcome,-0.042271,0.049597,NaN,NaN,NaN
Age,-0.028579,-0.046977,-0.031966,0.211885,0.221029
DiabetesPedigreeFunction,NaN,-0.055071,NaN,-0.166357,-0.153738
Insulin,NaN,NaN,-0.033826,NaN,NaN
BloodPressure,NaN,NaN,-0.032054,NaN,NaN
Pregnancies,NaN,NaN,NaN,0.170156,0.152681


### MNAR(Missing Not At Random) 비 무작위 결측판별
- 해당 변수의 결측 행을 다른 변수로 예측한 값과 관측치 예측값 격차 확인
- 예측된 값의 평균이 결측/관측 구간에서 크게 다르면(예: `mean_gap`이 다른 변수 대비 확연히 큰 경우) 값 자체가 특정 구간일 때 누락되는 MNAR 가능성을 의심할 수 있음
- 단, 데이터 수집 과정 상의 오류가 있었는지, 도메인 지식 관련 확인이 필요

In [9]:
mnar_signals = []
for col in target_cols:
    other_cols = [c for c in df2.columns if c != col]
    train_df = df2[df2[col].notna()]
    if train_df.empty:
        continue
    X_train = train_df[other_cols].fillna(train_df[other_cols].median())
    y_train = train_df[col]

    model = LinearRegression()
    model.fit(X_train, y_train)

    missing_part = df2[df2[col].isna()]
    if missing_part.empty:
        continue
    X_missing = missing_part[other_cols].fillna(df2[other_cols].median())
    X_observed = df2[df2[col].notna()][other_cols].fillna(df2[other_cols].median())

    pred_missing = model.predict(X_missing)
    pred_observed = model.predict(X_observed)

    mnar_signals.append({
        'variable': col,
        'pred_missing_mean': pred_missing.mean(),
        'pred_observed_mean': pred_observed.mean(),
        'mean_gap': abs(pred_missing.mean() - pred_observed.mean())
    })

pd.DataFrame(mnar_signals).set_index('variable')

,pred_missing_mean,pred_observed_mean,mean_gap
variable,,,
BMI,31.870733,32.457464,0.586731
BloodPressure,71.459522,72.404900,0.945378
Glucose,116.381480,121.686763,5.305283
Insulin,147.491820,155.532386,8.040566
SkinThickness,28.332799,29.151932,0.819132


## 3. 결측치 대체 전략 및 적용
- 전체 데이터를 훈련/검증/테스트로 분할 (Outcome를 기준으로 층화)하고 Imputation은 훈련/검증 세트에 대해서만 수행
- 현재 데이터에 Null 수가 적은 feature와 Null 수가 많은 feature가 혼재됨에 따라 4가지로 파일 및 변수를 생성 하여 `03_ML.ipynb`에서 ML 모델을 만드는데 훈련데이터, 검증을 위한 검증데이터, 실제 테스트 진행을 테스트 데이터로 진행
  - 중앙값 기반 단순 대체 훈련 데이터 (`diabetes_imputed_median_train.csv`)
  - 중앙값 기반 단순 대체 검증 데이터 (`diabetes_imputed_median_validation.csv`)
  - 중앙값 기반 단순 대체 테스트 데이터 (`diabetes_imputed_median_test.csv`)
  - Iterative Imputer(MICE 유사) 기반 다변량 대체 훈련 데이터(`diabetes_imputed_iterative_train.csv`)
  - Iterative Imputer(MICE 유사) 기반 다변량 대체 검증 데이터(`diabetes_imputed_iterative_validation.csv`)
  - Iterative Imputer(MICE 유사) 기반 다변량 대체 테스트 데이터(`diabetes_imputed_iterative_test.csv`)

In [10]:
from sklearn.model_selection import train_test_split

label_col = 'Outcome'
train_val, df_test = train_test_split(df2, test_size=0.2, random_state=42, stratify=df2[label_col])

# 0.25 * 0.8 = 0.2
df_train, df_val = train_test_split(train_val, test_size=0.25, random_state=42, stratify=train_val[label_col])

print(f"train: {df_train.shape}, val: {df_val.shape}, test: {df_test.shape}")

train: (460, 9), val: (154, 9), test: (154, 9)


In [11]:
# 훈련/검증 세트에서 0을 결측으로 변환 (EDA에서 측정 불가로 본 칼럼)
target_cols = ['BMI', 'BloodPressure', 'Glucose', 'Insulin', 'SkinThickness']

df_train_na = df_train.copy()
df_val_na = df_val.copy()
for subset in (df_train_na, df_val_na):
    subset[target_cols] = subset[target_cols].replace(0, pd.NA)

missing_train = pd.DataFrame({
    'missing_count': df_train_na[target_cols].isna().sum(),
    'missing_percentage': df_train_na[target_cols].isna().mean().round(4)
})
missing_val = pd.DataFrame({
    'missing_count': df_val_na[target_cols].isna().sum(),
    'missing_percentage': df_val_na[target_cols].isna().mean().round(4)
})

print('훈련 데이터 결측률')
display(missing_train)
print('검증 데이터 결측률')
display(missing_val)


훈련 데이터 결측률


,missing_count,missing_percentage
BMI,6,0.0130
BloodPressure,18,0.0391
Glucose,4,0.0087
Insulin,215,0.4674
SkinThickness,131,0.2848


검증 데이터 결측률


,missing_count,missing_percentage
BMI,3,0.0195
BloodPressure,5,0.0325
Glucose,0,0.0000
Insulin,75,0.4870
SkinThickness,44,0.2857


In [12]:
# 1) 중앙값 기반 단순 대체 (훈련+검증 기준)
train_val_medians = pd.concat([df_train, df_val])[target_cols].median()

imputed_train_median = df_train.copy()
imputed_val_median = df_val.copy()
imputed_test_median = df_test.copy()

for subset in (imputed_train_median, imputed_val_median, imputed_test_median):
    subset[target_cols] = subset[target_cols].fillna(train_val_medians)

# 2) Iterative Imputer (MICE 유사) - 훈련+검증으로 학습 후 전 세트 변환
train_val_clean = pd.concat([df_train, df_val]).replace({pd.NA: np.nan}).astype(float)
train_clean = df_train.replace({pd.NA: np.nan}).astype(float)
val_clean = df_val.replace({pd.NA: np.nan}).astype(float)
test_clean = df_test.replace({pd.NA: np.nan}).astype(float)

iter_imputer = IterativeImputer(random_state=42)
iter_imputer.fit(train_val_clean)

train_iter = iter_imputer.transform(train_clean)
val_iter = iter_imputer.transform(val_clean)
test_iter = iter_imputer.transform(test_clean)

imputed_train_iter = pd.DataFrame(train_iter, index=df_train.index, columns=df2.columns)
imputed_val_iter = pd.DataFrame(val_iter, index=df_val.index, columns=df2.columns)
imputed_test_iter = pd.DataFrame(test_iter, index=df_test.index, columns=df2.columns)

# 범주/정수형 칼럼을 원래 스케일로 정리
for int_col in ['Pregnancies', 'Outcome']:
    for df_part in (imputed_train_iter, imputed_val_iter, imputed_test_iter):
        if int_col in df_part.columns:
            df_part[int_col] = df_part[int_col].round().astype(int)

# 파일 저장
imputed_train_median.to_csv('data/diabetes_imputed_median_train.csv', index=False)
imputed_val_median.to_csv('data/diabetes_imputed_median_validation.csv', index=False)
imputed_test_median.to_csv('data/diabetes_imputed_median_test.csv', index=False)

imputed_train_iter.to_csv('data/diabetes_imputed_iterative_train.csv', index=False)
imputed_val_iter.to_csv('data/diabetes_imputed_iterative_validation.csv', index=False)
imputed_test_iter.to_csv('data/diabetes_imputed_iterative_test.csv', index=False)

print('저장 완료:')
print(' - data/diabetes_imputed_median_train.csv')
print(' - data/diabetes_imputed_median_validation.csv')
print(' - data/diabetes_imputed_median_test.csv')
print(' - data/diabetes_imputed_iterative_train.csv')
print(' - data/diabetes_imputed_iterative_validation.csv')
print(' - data/diabetes_imputed_iterative_test.csv')


저장 완료:
 - data/diabetes_imputed_median_train.csv
 - data/diabetes_imputed_median_validation.csv
 - data/diabetes_imputed_median_test.csv
 - data/diabetes_imputed_iterative_train.csv
 - data/diabetes_imputed_iterative_validation.csv
 - data/diabetes_imputed_iterative_test.csv
